# EnderLeaf script preparation

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

## Imports

In [ ]:
from pathlib import Path
import time
from itertools import product

from rich.pretty import pprint

import numpy as np
import cv2

from enderscope.scan_patterns import snake, plot_path
from enderscope.serial import list_ports, Stage
from enderscope.bed import bed
from enderscope.enderlights import Enderlights

import panel as pn

from enderleaf.image import Rectangle, lap_var, to_pil, safe_pil_resize
from enderleaf.tools import time_method
from enderleaf.preview_panel import preview
from enderleaf.qr_reader import get_qr_data
from enderleaf.tools import ensure_folder, format_datetime
from enderscope.enderlights_pi import Enderlights

In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

In [ ]:
%matplotlib widget

## Setup

## Constants

In [ ]:
TEMPLATE_LENGTH = 210
TEMPLATE_SIZE = (TEMPLATE_LENGTH, TEMPLATE_LENGTH)
ROW_COUNT, COL_COUNT = 9, 9
LEAF_DIAM = 17
CAM_RES = (4608, 2592)
CROP_TOP = 600
CROP_BOTTOM = 550
CROP_LEFT = 1300
CROP_RIGHT = 1470
COLUMNS = [i + 1 for i in range(COL_COUNT)]
ROWS = [chr(65 + i) for i in range(ROW_COUNT)]
DST_FLD = Path(".").joinpath("output")

## Scan Pattern

In [ ]:
positions = snake(cols=COL_COUNT, rows=ROW_COUNT) * [
    # steps
    TEMPLATE_LENGTH / COL_COUNT,
    TEMPLATE_LENGTH / ROW_COUNT,
] + [
    # origin
    TEMPLATE_LENGTH / COL_COUNT / 2,
    TEMPLATE_LENGTH / ROW_COUNT / 2,
]
plot_path(
    positions,
    title="snake scan",
    field=(LEAF_DIAM + 3, LEAF_DIAM + 3),
    selected_rectangles=[17, 41, 55, 81],
    circle_diam=LEAF_DIAM,
)

In [ ]:
positions[0]

## 3D Virtual Scan

In [ ]:
s = Stage("virtual", 115200)

In [ ]:
s.home()

In [ ]:
# s.move_position((bed.x_max / 2, bed.y_max / 2, bed.global_height))
for p in positions:
    s.move_position(np.append(p, bed.individual_height))

## 3D Scan

In [ ]:
def acquire_image(
    stage: Stage,
    lights: Enderlights,
    pos,
    crop_data: Rectangle | None = None,
):
    stage.move_position(pos)
    stage.finish_moves()
    lights.shutter(True)
    image = preview().do_capture_array(crop_data=crop_data)
    lights.shutter(False)
    return image

In [ ]:
def get_best_z(
    stage,
    lights,
    pos,
    crop_data: Rectangle,
    min_rel_z: float = 5.0,
    max_rel_z: float = 5.0,
):
    zrange = np.array(range(min_rel_z, max_rel_z, 1))
    mxScore = -1
    bestZ = 0
    lights.shutter(True)

    for z in zrange:
        stage.move_position([pos[0], pos[1], z + pos[2]])
        stage.finish_moves()
        img = preview().do_capture_array(crop_data=crop_data)
        grayImage = (
            np.float32(img[:, :, 0])
            + np.float32(img[:, :, 1])
            + np.float32(img[:, :, 2])
        ) / 3
        score = lap_var(grayImage)
        if score > mxScore:
            mxScore = score
            bestZ = z
    lights.shutter(False)

    stage.move_position([pos[0], pos[1], pos[2]])
    stage.finish_moves()

    return bestZ + pos[2]

In [ ]:
@time_method
def run_job(stage, lights, speed, start_height):
    # s.set_speed(speed=speed)
    crop_data = Rectangle(
        top=CROP_TOP,
        bottom=CAM_RES[1] - CROP_BOTTOM,
        left=CROP_LEFT,
        right=CAM_RES[0] - CROP_RIGHT,
    )

    z = get_best_z(
        stage=stage,
        lights=lights,
        pos=[*positions[0], start_height],
        crop_data=Rectangle(
            top=CROP_TOP,
            bottom=CAM_RES[1] - CROP_BOTTOM,
            left=CROP_LEFT,
            right=CAM_RES[0] - CROP_RIGHT,
        ),
        min_rel_z=-5,
        max_rel_z=5,
    )

    qr_data = get_qr_data(
        acquire_image(
            stage=stage, lights=lights, pos=[*positions[0], z], crop_data=crop_data
        )
    )
    exp_name = qr_data["info"][0] if qr_data["retval"] is True else "UNKNOWN"

    image_data = []
    ensure_folder(DST_FLD.joinpath(exp_name))

    for p, (c, r) in zip(positions, list(product(COLUMNS, ROWS))):
        image = acquire_image(
            stage=stage, lights=lights, pos=np.append(p, z), crop_data=crop_data
        )
        file_name = f"{exp_name}#{r}#{c}#{format_datetime()}"
        cv2.imwrite(
            str(DST_FLD.joinpath(exp_name).joinpath(file_name).with_suffix(".png")),
            image,
        )
        image_data.append(
            {
                "exp_name": exp_name,
                "row": r,
                "col": c,
                "image": image,
                "position": p,
                "file_name": file_name,
            }
        )

    stage.move_position((TEMPLATE_LENGTH / 2, TEMPLATE_LENGTH / 2, 100))
    return image_data

In [ ]:
# list available serial ports
stage_port = None
ports = list_ports()
for port in ports:
    if "USB Serial" in port.description:
        stage_port = port
        print("* " + str(port))
    else:
        print("  " + str(port))

In [ ]:
stage = Stage(stage_port, 115200)

In [ ]:
lights = Enderlights()
lights.shutter(True)
time.sleep(1)
lights.shutter(False)

In [ ]:
success = stage.safe_home()
if success:
    stage.move_position((TEMPLATE_LENGTH / 2, TEMPLATE_LENGTH / 2, 100))
    stage.finish_moves()
else:
    raise ConnectionError("Unable to home")

In [ ]:
preview().start_still()
preview().set_crop(top=CROP_TOP, bottom=CROP_BOTTOM, left=CROP_LEFT, right=CROP_RIGHT)

In [ ]:
preview().show()

In [ ]:
preview().camera.set_controls(
    {"LensPosition": preview().camera.camera_controls["LensPosition"][1]}
)

In [ ]:
images = run_job(stage=stage, lights=lights, speed=6000, start_height=36)
len(images)

In [ ]:
sel_image = pn.widgets.IntSlider(
    name="Select image",
    start=0,
    end=len(images) - 1,
    value=0,
    sizing_mode="stretch_width",
)
ph_image = pn.pane.Placeholder()
json_data = pn.pane.JSON()


@pn.depends(sel_image.param.value, watch=True)
def on_index_changed(index):
    ph_image.object = safe_pil_resize(to_pil(images[index]["image"]), 600,600)
    json_data.object = {k:str(v) for k, v in images[index].items() if k != "image"}


on_index_changed(sel_image.value)

pn.Column(ph_image, json_data, sel_image)

In [ ]:
get_qr_data(images[0])

In [ ]:
from PIL import ImageOps

ImageOps.contain(to_pil(images[0]), (204,204))

In [ ]:
safe_pil_resize(to_pil(images[0]), 300,300)

In [ ]:
stage.set_speed_limit(100000, debug=True)

In [ ]:
stage.write_code("M503", debug=True)

In [ ]:
stage.set_speed(6000)
stage.move_relative(-100, -100)
stage.move_relative(100, 100)

In [ ]:
stage.move_axis("z", -50)

In [ ]:
from timeit import default_timer as timer
import time

before = timer()
time.sleep(1)
after = timer()

after - before